In [7]:
import os
from natsort import natsorted
import random
import torch
import torchvision
from skimage import io
from PIL import Image
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
from skimage.feature import hog
from skimage import color, exposure
from skimage.color import rgb2gray
from scipy.stats import pearsonr
import numpy as np

os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

no_of_examples = 200

# This method selects the pixels in a radius (half of the width or height) around the center
# ignores the rest of the pixels and also uses RGB info
def masked_corr_coef(img_a, img_b):
    
    img_a = np.array(img_a)
    img_b = np.array(img_b)
    
    width_a = img_a.shape[0]  
    height_a = img_a.shape[1]
    width_b = img_b.shape[0]  
    height_b = img_b.shape[1]
   
    # reduced image does not have pixels outside the defined radius
    reduced_img_a = []
    reduced_img_b = []
    
    if width_a == width_b and width_a == height_a :
          
        # center and radius of the circle
        center_x = width_a//2 # floor divsion (to a whole number)
        center_y = height_a//2
        radius = min(center_x, center_y)
        
        # Iterate through all pixels in the image
        for x in range(width_a):
            for y in range(height_a):
                # Calculate the distance from the center of the circle
                distance = ((x - center_x) ** 2 + (y - center_y) ** 2) ** 0.5

                # Check if the distance is within the circle's radius
                if distance <= radius:
                   # Get the pixel color from the original image
                    pixel_color_a = img_a[x,y,:]
                    pixel_color_b = img_b[x,y,:]
                    reduced_img_a.append(pixel_color_a)
                    reduced_img_b.append(pixel_color_b)
                    
        reduced_img_a = np.array(reduced_img_a)
        reduced_img_b = np.array(reduced_img_b) 
        
        corr_coeff, _ = pearsonr(reduced_img_a.flatten(),reduced_img_b.flatten())
        return corr_coeff
#         print('Size of reduced image',reduced_img_a.shape)
#         print('Size of original image',img_a.shape)
                   
 

                

def display_images_side_by_side(images_side_by_side):
    num_images = len(images_side_by_side)
    fig, axes = plt.subplots(1, num_images, figsize=(10, 5))
    for i in range(num_images):
        image = images_side_by_side[i]
      
        axes[i].imshow(image)
        axes[i].axis('off')    
    plt.tight_layout()
    plt.show()

path_name = r"C:\Users\psh006\OneDrive - UiT Office 365\ML codes\AICE\CorrespondingImages"

# List of folder names for the patients
Pfolder_names = os.listdir(path_name)

P_names_list = ['Patient_0001','Patient_0001','Patient_0003','Patient_0003']
H_names_list = ['Head_1','Head_2','Head_1','Head_2']

SubFolder_names = []

path_name = r"C:\Users\psh006\OneDrive - UiT Office 365\ML codes\AICE\CorrespondingImages"

# List of folder names for the patients
Pfolder_names = os.listdir(path_name)

head_folder_names = [r"Head_1",r"Head_2"]
SubFolder_names = []
P_names = []
H_names = []
Cluster_names = []
no_of_folders_list = []
for ls in Pfolder_names:
    for sh in head_folder_names:        
        
        # Join paths and find the subfolders names where data is present
        str = os.path.join(path_name,ls,sh)
        names_of_subFolders = os.listdir(str)
        #print(len(names_of_subFolders))
       
        # Sorted files 
        names_of_subFolders = natsorted(names_of_subFolders) 
        
        max_no_of_folders = len(names_of_subFolders)
        #print(len(names_of_subFolders))
        no_of_folders_list.append(len(names_of_subFolders))
        for i in range(max_no_of_folders):
            
            #print(os.path.join(path_name,ls,sh,names_of_subFolders[i]))
            f_no= i # folder number (use this integer value to find non-corresponding images later)
            SubFolder_names.append(os.path.join(path_name,ls,sh,names_of_subFolders[i]))
            Cluster_names.append(f_no)
            P_names.append(ls)
            H_names.append(sh)

            
print('Total length of Subfolders is = ',len(SubFolder_names)) 
            

InterClassCC = []
IntraClassCC = []       

# convert a file no to a cluster number by normalizing
for i in range(no_of_examples):
    
    

    
    corres_f_no = random.randint(0,len(SubFolder_names))
    # head and tail pair
    head_tail = os.path.split(SubFolder_names[corres_f_no])
    #print(head_tail[0],head_tail[1])
    #List all files in the folder
    jpeg_list = os.listdir(SubFolder_names[corres_f_no]) 
    
    #print(len(jpeg_list))
    
    print('This is the folder number with N corresponding images = ',corres_f_no)
    
    corres_img_list = []
    non_corres_img_list = []
    
    # Find N (10) non corresponding folder numbers
    for j in range(len(jpeg_list)):
        
        corres_path_name = os.path.join(SubFolder_names[corres_f_no],jpeg_list[j])
        corres_img = io.imread(corres_path_name) 
        corres_img_list.append(corres_img)
        
        if corres_f_no < 200:
            temp_f_no = random.randint(corres_f_no+40,corres_f_no+100)
        elif corres_f_no> 200 and corres_f_no < len(SubFolder_names)-200:
            temp_f_no = random.randint(corres_f_no+40,corres_f_no+100)
        elif corres_f_no > len(SubFolder_names)-200:
            temp_f_no = random.randint(corres_f_no-200,corres_f_no-40)
        
        neigh_jpeg_list = os.listdir(SubFolder_names[temp_f_no])
        rand_jpeg_indx = random.randint(0, 9)
        neigh_file_path = os.path.join(SubFolder_names[temp_f_no],neigh_jpeg_list[rand_jpeg_indx])
        non_corres_img = io.imread(neigh_file_path) 
        non_corres_img_list.append(non_corres_img)
    
    for ii in range(0,10):
       
        #display_images_side_by_side([non_corres_img_list[ii],corres_img_list[ii]])
        # correlation between non-corresponding images
        correlation_coefficient_inter = masked_corr_coef(corres_img_list[ii],non_corres_img_list[ii])
        InterClassCC.append(correlation_coefficient_inter)
        
        if ii>0:
            # correlation within a set of corresponding images
            #correlation_coefficient_intra, _ = pearsonr(corres_img_list[ii].flatten(),corres_img_list[ii-1].flatten())
            correlation_coefficient_intra = masked_corr_coef(corres_img_list[ii], corres_img_list[ii-1])
            IntraClassCC.append(correlation_coefficient_intra)
            
       
        
    
        


# Convert the list to a NumPy array
InterClassCC_arr = np.array(InterClassCC)
# Convert the list to a NumPy array
IntraClassCC_arr = np.array(IntraClassCC)

print(InterClassCC_arr.size)
print(IntraClassCC_arr.size)

print('Mean Correlation Between Non Corresponding Images = ',np.mean(InterClassCC_arr))
print('Mean Correlation Within Corresponding Images =',np.mean(IntraClassCC_arr))
print('Std of Correlations Between Non Corresponding Images = ',np.std(InterClassCC_arr))
print('Std of Correlations  Within Corresponding Images  =',np.std(IntraClassCC_arr))
    

    








Total length of Subfolders is =  1243
This is the folder number with N corresponding images =  440
This is the folder number with N corresponding images =  258
This is the folder number with N corresponding images =  449
This is the folder number with N corresponding images =  279
This is the folder number with N corresponding images =  121
This is the folder number with N corresponding images =  1128
This is the folder number with N corresponding images =  974
This is the folder number with N corresponding images =  993
This is the folder number with N corresponding images =  814
This is the folder number with N corresponding images =  1167
This is the folder number with N corresponding images =  884
This is the folder number with N corresponding images =  1113
This is the folder number with N corresponding images =  107
This is the folder number with N corresponding images =  861
This is the folder number with N corresponding images =  615
This is the folder number with N correspondi

This is the folder number with N corresponding images =  718
This is the folder number with N corresponding images =  430
This is the folder number with N corresponding images =  994
This is the folder number with N corresponding images =  42
This is the folder number with N corresponding images =  186
This is the folder number with N corresponding images =  481
This is the folder number with N corresponding images =  918
This is the folder number with N corresponding images =  696
This is the folder number with N corresponding images =  354
This is the folder number with N corresponding images =  914
This is the folder number with N corresponding images =  613
This is the folder number with N corresponding images =  1047
This is the folder number with N corresponding images =  932
This is the folder number with N corresponding images =  639
This is the folder number with N corresponding images =  962
This is the folder number with N corresponding images =  341
This is the folder numbe